In [2]:
from tensorboardX import SummaryWriter

import os
import gc
import shutil
import numpy as np
import pandas as pd
import pickle

import sys
import warnings
sys.path.append('models')
sys.path.append('utils')
warnings.filterwarnings("ignore")

import torch
import torch.optim as optim
from torch.utils.data.dataloader import default_collate
from sksurv.metrics import concordance_index_censored

from torch.utils.data import DataLoader, Dataset

## Prove Iniziali

In [4]:
os.environ["PYDEVD_WARN_SLOW_RESOLVE_TIMEOUT"] = "2"

dataHum = pd.read_csv('../dataset/MultiomicsFinal.csv', index_col='ID')
multiomics = pd.read_csv('../dataset_multiomics/Input_MM_ICH_top2000_norm_Scaled.csv', index_col='ID')

In [5]:
def custom_collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    return default_collate(batch)

In [ ]:
def seed_torch(device, seed=42):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda:0':
    torch.cuda.empty_cache()
    gc.collect()
print('Device:', device)

seed_torch(seed=42, device=device)
modifed_data = create_bins(data=dataHum, label_col=None, n_bins=4, eps=1e-6)

dataset_dict = {}
for file in os.listdir('../dataset_multiomics'):
    if file.endswith('.pickle'):
        with open(f"../dataset_multiomics/{file}", 'rb') as f:
            pickle_data = pickle.load(f)
        
        for key in pickle_data.keys():
            if key not in dataset_dict.keys():
                dataset_dict[key] = pickle_data[key]
                dataset_dict[key]['time'] = modifed_data.loc[key, 'OS censored at TPX  months']

Device: cuda:0


In [5]:
if not os.path.exists(os.path.join('results_multiomics', str(0), f'cv_{0}')):
    os.makedirs(os.path.join('results_multiomics', str(0), f'cv_{0}'), exist_ok=True)
else:
    shutil.rmtree(os.path.join('results_multiomics', str(0)))
    os.makedirs(os.path.join('results_multiomics', str(0), f'cv_{0}'), exist_ok=True)

writer = SummaryWriter(os.path.join('results_multiomics', str(0), f'cv_{0}'), flush_secs=15)

print(f'\nCross Validation: Fold {0}')
split_data = pd.read_csv(f'splits/0.csv')
ids_train = split_data['train']
ids_val = split_data['val']

if ids_val.dtype == 'float64':
    ids_val = ids_val.astype('Int64')

if ids_train.dtype == 'float64':
    ids_train = ids_train.astype('Int64')

train_dict = {patient_id: dataset_dict[patient_id] for patient_id in ids_train if patient_id in dataset_dict and '_dp' not in patient_id}
val_dict = {patient_id: dataset_dict[patient_id] for patient_id in ids_val if patient_id in dataset_dict and '_dp' not in patient_id}

train_dataset = MultiomicsDataset(train_dict, rows=15000)
val_dataset = MultiomicsDataset(val_dict, rows=15000)

train_loader = DataLoader(train_dataset, batch_size=21, collate_fn=custom_collate_fn, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=21, collate_fn=custom_collate_fn, shuffle=False, drop_last=True)


Cross Validation: Fold 0


In [6]:
mode = 'w'
filename = f"results_0.csv"

csv_folder = os.path.join('results_multiomics', str(0))

model_img = ModelImages(input_size=1536, surv_nodes=[1536, 64, 1], dropout=0.3)
model_img.to(device)

model_omics = OtherOmicsModel(input_size=2061, transc_nodes=[2061, 64, 1], dropout=0.3)
model_omics.to(device)

criterion_images = cox_ph_loss
criterion_omics = cox_ph_loss
optimizer_images = optim.Adam(model_img.parameters(), lr=1e-05, weight_decay=1e-04)
optimizer_omics = optim.Adam(model_omics.parameters(), lr=1e-05, weight_decay=1e-04)

scheduler_images = optim.lr_scheduler.CosineAnnealingLR(optimizer_images, T_max = 100, eta_min = 0, last_epoch = -1)
scheduler_omics = optim.lr_scheduler.CosineAnnealingLR(optimizer_omics, T_max = 100, eta_min = 0, last_epoch = -1)

In [6]:
seed_torch(seed=42, device=device)

## Find Parameters

In [11]:
import itertools

params = {
    'objective': ['survival:cox'],
    'eval_metric': ['cox-nloglik'],
    'eta': [0.1, 0.05, 0.01],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'lambda': [1.0],
    'alpha': [0],
    'gamma': [0],
    'min_child_weight': [1],
    'scale_pos_weight': [1, 2],
}

keys, values = zip(*params.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
c_index_final = 0.5
params_final = None


for i, combo in enumerate(combinations):
    total_loss_img = 0
    total_loss_omics = 0

    all_risk_img_scores = []
    all_risk_omics_scores = []
    all_censorships = []
    all_real_times = []

    demog_complete = []
    clin_complete = []
    trascr_complete = []

    for train_sample in train_loader:
        features, demog, clin, genomic, transcr, real_time, event_indicator, masking = train_sample['staining'], train_sample['demog'], train_sample['clin'], train_sample['genomic'], train_sample['transcr'], train_sample['time'], train_sample['outcome'], train_sample['masking']

        all_censorships.append(event_indicator.numpy())
        all_real_times.append(real_time.numpy())

        demog_complete.append(demog)
        clin_complete.append(clin)
        trascr_complete.append(transcr)

        torch.cuda.empty_cache()

    all_censorships = np.concatenate(all_censorships)
    all_real_times = np.concatenate(all_real_times)

    X_cox = torch.cat([torch.cat(demog_complete, dim=0), torch.cat(clin_complete, dim=0), torch.cat(trascr_complete, dim=0)], dim=1).numpy()
    y_cox = np.array([(bool(e), t) for e, t in zip(all_censorships, all_real_times)], dtype=[('event', '?'), ('time', '<f8')])

    dtrain = xgb.DMatrix(X_cox, label=all_real_times, base_margin=all_censorships)

    xgb_model = xgb.train(combo, dtrain, num_boost_round=100)
    all_risk_cox = xgb_model.predict(dtrain)

    try:
        c_index = concordance_index_censored((all_censorships.reshape(-1,)).astype(bool), all_real_times, all_risk_cox.reshape(-1,), tied_tol=1e-08)[0]
    except Exception as e:
        print(e)
        c_index = 0.5

    eval_loss_img = 0
    eval_loss_omics = 0

    eval_risk_img_scores = []
    eval_risk_omics_scores = []
    eval_censorships = []
    eval_real_times = []

    demog_val_complete = []
    clin_val_complete = []
    trascr_val_complete = []

    for val_sample in val_loader:
        features_val, demog_val, clin_val, genomic_val, transcr_val, real_times_val, event_indicator_val, masking_val = val_sample['staining'], val_sample['demog'], val_sample['clin'], val_sample['genomic'], val_sample['transcr'], val_sample['time'], val_sample['outcome'], val_sample['masking']

        eval_censorships.append(event_indicator_val.numpy())
        eval_real_times.append(real_times_val.numpy())

        demog_val_complete.append(demog_val)
        clin_val_complete.append(clin_val)
        trascr_val_complete.append(transcr_val)

        torch.cuda.empty_cache()

    eval_censorships = np.concatenate(eval_censorships)
    eval_real_times = np.concatenate(eval_real_times)

    x_val_cox = torch.cat([torch.cat(demog_val_complete, dim=0), torch.cat(clin_val_complete, dim=0), torch.cat(trascr_val_complete, dim=0)], dim=1).numpy()
    dtest = xgb.DMatrix(x_val_cox)

    eval_risk_cox = xgb_model.predict(dtest)    

    try:
        c_index_val = concordance_index_censored((eval_censorships.reshape(-1,)).astype(bool), eval_real_times, eval_risk_cox.reshape(-1,), tied_tol=1e-08)[0]
    except Exception as e:
        print(e)
        c_index_val = 0.5

    if c_index_val > c_index_final:
        print('Train_c_index: {:.4f}, Val_c_index: {:.4f}'.format(c_index, c_index_val))
        c_index_final = c_index_val
        params_final = combo

    # if (epoch + 1) % 2 == 0:
    #     print('Epoch: {}, Train_c_index: {:.4f}, Val_c_index: {:.4f}'.format(epoch + 1, c_index, c_index_val))

Train_c_index: 0.9825, Val_c_index: 0.7977
Train_c_index: 0.9821, Val_c_index: 0.8085
Train_c_index: 0.9815, Val_c_index: 0.8345


KeyboardInterrupt: 

In [12]:
params_final

{'objective': 'survival:cox',
 'eval_metric': 'cox-nloglik',
 'eta': 0.1,
 'max_depth': 3,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'lambda': 1.0,
 'alpha': 0,
 'gamma': 0,
 'min_child_weight': 1,
 'scale_pos_weight': 2}

## Predizione Finale

In [2]:
import xgboost as xgb

from sksurv.metrics import concordance_index_censored
from itertools import product

from sklearn.model_selection import StratifiedKFold
from models_omics import ModelImages

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

In [3]:
dataHum = pd.read_csv('dataset_new/MultiomicsFinal.csv', index_col='ID')
multiomics = pd.read_csv('dataset_multiomics/Input_MM_ICH_top2000_norm_Scaled.csv', index_col='ID')

In [4]:
model_img = ModelImages(input_size=1536, surv_nodes=[1536, 64, 1], dropout=0.3)
model_img.load_state_dict(torch.load("matrix/model_img_0.pth"))
model_img

ModelImages(
  (attention_net): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=1536, out_features=1536, bias=True)
  )
  (surv_layers): ModuleList(
    (0): Linear(in_features=1536, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=1, bias=True)
  )
  (dropout_coxnet): Dropout(p=0.3, inplace=False)
)

In [4]:
def custom_collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    return default_collate(batch)

In [5]:
seed_torch(seed=42, device=device)
modifed_data = create_bins(data=dataHum, label_col=None, n_bins=4, eps=1e-6)

dataset_dict = {}
for file in os.listdir('dataset_multiomics'):
    if file.endswith('.pickle'):
        with open(f"dataset_multiomics/{file}", 'rb') as f:
            pickle_data = pickle.load(f)
        
        for key in pickle_data.keys():
            if key not in dataset_dict.keys():
                dataset_dict[key] = pickle_data[key]
                dataset_dict[key]['time'] = modifed_data.loc[key, 'OS censored at TPX  months']

In [6]:
split_data = pd.read_csv(f'splits/0.csv')
ids_train = split_data['train']
ids_val = split_data['val']

if ids_val.dtype == 'float64':
    ids_val = ids_val.astype('Int64')

if ids_train.dtype == 'float64':
    ids_train = ids_train.astype('Int64')

train_dict = {patient_id: dataset_dict[patient_id] for patient_id in ids_train if patient_id in dataset_dict and '_dp' not in patient_id}
val_dict = {patient_id: dataset_dict[patient_id] for patient_id in ids_val if patient_id in dataset_dict and '_dp' not in patient_id}

train_dataset = MultiomicsDataset(train_dict, rows=15000)
val_dataset = MultiomicsDataset(val_dict, rows=15000)

train_loader = DataLoader(train_dataset, batch_size=21, collate_fn=custom_collate_fn, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=21, collate_fn=custom_collate_fn, shuffle=False, drop_last=True)

In [8]:
demog_complete = []
clin_complete = []
genomic_complete = []
transcr_complete = []
representation_complete = []

all_time = []
all_event = []

model_img.eval()
with torch.no_grad():
    for train_sample in train_loader:
        features, demog, clin, genomic, transcr, real_time, event_indicator, masking = train_sample['staining'], train_sample['demog'], train_sample['clin'], train_sample['genomic'], train_sample['transcr'], train_sample['time'], train_sample['outcome'], train_sample['masking']
        _, representation_train = model_img(features, masking)
        
        demog_complete.append(demog)
        clin_complete.append(clin)
        genomic_complete.append(genomic)
        transcr_complete.append(transcr)

        all_time.append(real_time)
        all_event.append(event_indicator)

        representation_complete.append(representation_train.detach().numpy())
        torch.cuda.empty_cache()

In [9]:
all_demog = np.concatenate(demog_complete)
all_clin = np.concatenate(clin_complete)
all_genomic = np.concatenate(genomic_complete)
all_transcr = np.concatenate(transcr_complete)
z_train = np.concatenate(representation_complete)

X_train = np.concatenate([all_demog, all_clin, all_genomic, all_transcr, z_train], axis=1)
np.save('matrix/X_train.npy', X_train)
np.save('matrix/t_train.npy', np.concatenate(all_time))
np.save('matrix/e_train.npy', np.concatenate(all_event))

In [7]:
model_img = ModelImages(input_size=1536, surv_nodes=[1536, 64, 1], dropout=0.3)
model_img.load_state_dict(torch.load("matrix/model_img_0.pth"))
model_img

ModelImages(
  (attention_net): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=1536, out_features=1536, bias=True)
  )
  (surv_layers): ModuleList(
    (0): Linear(in_features=1536, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=1, bias=True)
  )
  (dropout_coxnet): Dropout(p=0.3, inplace=False)
)

In [8]:
demog_val_complete = []
clin_val_complete = []
genomic_val_complete = []
transcr_val_complete = []
representation_val_complete = []

all_time_val = []
all_event_val = []

model_img.eval()
with torch.no_grad():
    for val_sample in val_loader:
        features_val, demog_val, clin_val, genomic_val, transcr_val, real_times_val, event_indicator_val, masking_val = val_sample['staining'], val_sample['demog'], val_sample['clin'], val_sample['genomic'], val_sample['transcr'], val_sample['time'], val_sample['outcome'], val_sample['masking']
        _, representation_val = model_img(features_val, masking_val)

        demog_val_complete.append(demog_val)
        clin_val_complete.append(clin_val)
        genomic_val_complete.append(genomic_val)
        transcr_val_complete.append(transcr_val)

        all_time_val.append(real_times_val)
        all_event_val.append(event_indicator_val)

        representation_val_complete.append(representation_val.detach().numpy())
        torch.cuda.empty_cache()

In [9]:
all_demog_val = np.concatenate(demog_val_complete)
all_clin_val = np.concatenate(clin_val_complete)
all_genomic_val = np.concatenate(genomic_val_complete)
all_transcr_val = np.concatenate(transcr_val_complete)
z_val = np.concatenate(representation_val_complete)

X_test = np.concatenate([all_demog_val, all_clin_val, all_genomic_val, all_transcr_val, z_val], axis=1)
np.save('matrix/X_test.npy', X_test)
np.save('matrix/t_test.npy', np.concatenate(all_time_val))
np.save('matrix/e_test.npy', np.concatenate(all_event_val))